In [ ]:
# Batch classify all CFTR2 variants from the Scientific page

%pip install selenium beautifulsoup4 pandas

import os
import re
import csv
import json
import time
import traceback
import urllib.parse
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# -----------------------------
# Helpers
# -----------------------------

def safe_filename(variant: str) -> str:
    return re.sub(r'[<>:"/\\|?*]', "_", variant)

def double_encode(variant: str) -> str:
    # CFTR2 needs double percent-encoding for URLs
    return urllib.parse.quote(urllib.parse.quote(variant, safe=""), safe="")

def encode_variant(variant: str) -> str:
    # kept for parity with your earlier code
    return double_encode(variant)


# -----------------------------
# Scrape one scientific page (session already established)
# -----------------------------

def fetch_scientific_page_html(driver, wait, variant: str, save_folder: str) -> str:
    encoded = encode_variant(variant)
    url = f"https://cftr2.org/mutation/scientific/{encoded}"
    print(f"🔗 [{variant}] Scientific URL: {url}")
    driver.get(url)

    # retry if the site takes time to route
    reached = False
    for attempt in range(3):
        time.sleep(2.5)
        if "scientific" in driver.current_url.lower():
            reached = True
            break
        print(f"🔁 [{variant}] Not on scientific page yet (attempt {attempt+1})")
        driver.get(url)

    if not reached:
        raise RuntimeError("Could not reach scientific page after retries")

    # Activate Functional Testing tab if present
    try:
        func_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(), 'Functional Testing')]")))
        driver.execute_script("arguments[0].scrollIntoView(true);", func_tab)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", func_tab)
        time.sleep(3)
    except Exception as e:
        # Attempt a direct find/click fallback
        try:
            func_tab = driver.find_element(By.XPATH, "//a[contains(text(), 'Functional Testing')]")
            driver.execute_script("arguments[0].scrollIntoView(true);", func_tab)
            time.sleep(0.5)
            func_tab.click()
            time.sleep(3)
        except Exception as e2:
            print(f"⚠️ [{variant}] Could not activate Functional Testing tab: {e2}")

    os.makedirs(save_folder, exist_ok=True)
    html_path = os.path.join(save_folder, f"{safe_filename(variant)}.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(driver.page_source)
    return html_path


# -----------------------------
# Parse scientific HTML
# -----------------------------

def parse_cftr2_html(file_path):
    import re
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    # Variant type text
    variant_type_text = ""
    for h4 in soup.find_all("h4"):
        if "functional testing for" in h4.text.lower():
            next_p = h4.find_next_sibling("p")
            if next_p and len(next_p.text.strip()) < 500:
                variant_type_text = next_p.text.strip()
            break

    if not variant_type_text:
        for tag in soup.find_all(["p", "li", "div", "td"]):
            if any(term in tag.text.lower() for term in [
                "splice", "splicing", "splice site", "splice donor", "splice acceptor",
                "splice junction", "splice variant", "aberrant splicing"
            ]):
                candidate = tag.text.strip()
                if len(candidate) < 500:
                    variant_type_text = candidate
                    break

    # Functional data
    function_data = []
    for table in soup.find_all("table", class_="missense_table"):
        for row in table.find_all("tr")[1:]:
            cols = row.find_all("td")
            if len(cols) >= 4:
                cell_line = cols[1].text.strip()
                quantity_text = cols[2].text.strip()
                function_text = cols[3].text.strip()

                def try_float(text):
                    m = re.search(r"\d+\.?\d*", text)
                    return float(m.group()) if m else None

                function_data.append({
                    "cell_line": cell_line,
                    "quantity": try_float(quantity_text),
                    "function": try_float(function_text)
                })

    # Clinical data (the small table on the page)
    clinical_data = {}
    variant_col_name = None
    clinical_table = soup.find("table", class_="clinical")
    if clinical_table:
        rows = clinical_table.find_all("tr")
        if rows:
            headers = [h.get_text(strip=True) for h in rows[0].find_all(["th", "td"])]
            if len(headers) >= 3:
                variant_col_name = headers[1] or "Variant"
            for row in rows[1:]:
                cells = row.find_all("td")
                if len(cells) >= 3:
                    key = cells[0].get_text(strip=True)
                    clinical_data[key] = {
                        (variant_col_name or "Variant"): cells[1].get_text(strip=True),
                        "CF Combo": cells[2].get_text(strip=True)
                    }

    # FEV1 tabular chart
    fev1_data = []
    for div in soup.find_all("div", attrs={"aria-label": re.compile("tabular.*chart", re.I)}):
        table = div.find("table")
        if table:
            for row in table.find_all("tr")[1:]:
                cols = row.find_all("td")
                if len(cols) >= 2:
                    label = cols[0].text.strip()
                    try:
                        value = float(cols[1].text.strip())
                    except:
                        value = None
                    fev1_data.append((label, value))

    return function_data, clinical_data, fev1_data, variant_type_text


# -----------------------------
# Clinical helpers
# -----------------------------

def _to_float_safe(x):
    if x is None:
        return None
    x = str(x).strip()
    x = x.replace('%', '').replace(',', '').strip()
    if x.startswith('<') or x.startswith('>'):
        x = x[1:]
    m = re.search(r"\d+\.?\d*", x)
    if not m:
        return None
    try:
        return float(m.group())
    except:
        return None

def assess_clinical_severity(clinical_data, variant_name=None):
    def pick_value(row_dict):
        if not row_dict:
            return None
        if variant_name and variant_name in row_dict:
            return _to_float_safe(row_dict.get(variant_name))
        for k, v in row_dict.items():
            if k != "CF Combo":
                val = _to_float_safe(v)
                if val is not None:
                    return val
        return _to_float_safe(row_dict.get("CF Combo"))

    details = {}
    flags = set()
    score = 0

    sweat = pick_value(clinical_data.get("Sweat Chloride (non-CF is less than 30 mEq/L)", {}))
    details["sweat_chloride_cf_combo"] = sweat
    if sweat is not None:
        if sweat > 90:
            score += 2; flags.add("sweat_high")
        elif sweat > 60:
            score += 1; flags.add("sweat_moderate")

    pi = pick_value(clinical_data.get("Pancreatic Insufficiency (less than 1% of non-CF expected to be PI)", {}))
    details["pi_cf_combo_pct"] = pi
    if pi is not None:
        if pi >= 80:
            score += 2; flags.add("pi_high")
        elif pi >= 50:
            score += 1; flags.add("pi_moderate")

    pseudo = pick_value(clinical_data.get("Pseudomonas (less than 1% of non-CF expected to have Pseudomonas)", {}))
    details["pseudomonas_cf_combo_pct"] = pseudo
    if pseudo is not None and pseudo >= 40:
        score += 1; flags.add("pseudomonas_common")

    fev_bins = ["<10", "10-20", ">20"]
    fev_vals = []
    for b in fev_bins:
        val = pick_value(clinical_data.get(b, {}))
        if val is not None:
            fev_vals.append(val)
            details[f"fev1_{b}_cf_combo_pct"] = val
    if fev_vals:
        fev_mean = sum(fev_vals) / len(fev_vals)
        details["fev1_mean_cf_combo_pct"] = fev_mean
        if fev_mean < 40:
            score += 2; flags.add("fev1_severely_reduced")
        elif fev_mean < 60:
            score += 1; flags.add("fev1_reduced")

    age = pick_value(clinical_data.get("Average Age", {}))
    details["avg_age_cf_combo"] = age

    return score, details, flags

def describe_clinical_support(score, flags):
    if score >= 5:
        return "clinical_severe"
    if score >= 3:
        return "clinical_moderate"
    return "clinical_mild"


# -----------------------------
# Classifier
# -----------------------------

def classify_variant(function_data, clinical_data, fev1_data, variant_type, variant_name=None):
    variant_type = (variant_type or "").strip()
    variant_type_lower = variant_type.lower()
    predicted_class = None

    known_class5_splice = {"3849+10kbC->T", "2789+5G->A", "3272-26A->G"}
    borderline_iv_v = {"R117C", "R347H"}

    # If no functional data rows or all empty
    if (not function_data) or all(d.get("quantity") is None and d.get("function") is None for d in function_data):
        avg_quantity = 0.0
        avg_function = 0.0

        if variant_name in known_class5_splice:
            return "Class V (Residual function - splice variant)", avg_quantity, avg_function, {
                "clinical_reconciliation": None,
                "clinical_score": None,
                "clinical_flags": [],
                "explanation": "Known Class V splice override without functional data."
            }

        soup = BeautifulSoup(variant_type, 'html.parser')
        bold_texts = [b.get_text().lower() for b in soup.find_all('b')]
        has_functional_bold = any("quantity" in t or "function" in t for t in bold_texts)

        splice_keywords = ["splice donor", "splice acceptor", "splicing defect", "disrupts mrna processing",
                           "variant to the splice", "splice site", "affects splicing", "splice junction",
                           "splice variant", "aberrant splicing"]
        processing_keywords = ["processing defect", "misfolded", "misprocessing", "trafficking defect",
                               "class ii", "defective processing", "protein folding"]
        nonsense_keywords = ["nonsense", "stop gain", "premature termination", "stop-gain", "stop codon"]

        if any(k in variant_type_lower for k in nonsense_keywords):
            predicted_class = "Class I (No synthesis - inferred from text)"
        elif any(k in variant_type_lower for k in splice_keywords):
            predicted_class = "Class V (Residual function - splice variant)"
        elif any(k in variant_type_lower for k in processing_keywords):
            predicted_class = "Class II (Defective processing - inferred)"
        elif not has_functional_bold:
            predicted_class = "Uncertain class (no functional data)"
        else:
            predicted_class = "Uncertain class (bolded parameters present but no structured data)"

        c_score, c_details, c_flags = assess_clinical_severity(clinical_data or {})
        return predicted_class, avg_quantity, avg_function, {
            "clinical_reconciliation": describe_clinical_support(c_score, c_flags),
            "clinical_score": c_score,
            "clinical_flags": sorted(list(c_flags)),
            "explanation": "Predicted from text; functional data unavailable."
        }

    # Filter and average per cell line
    filtered_data = [
        d for d in function_data
        if d.get("cell_line", "").lower() not in ["all cell lines", "cell lines"]
        and d.get("quantity") is not None
        and d.get("function") is not None
    ]
    cell_line_map = {}
    for d in filtered_data:
        key = d["cell_line"]
        cell_line_map.setdefault(key, []).append((d["quantity"], d["function"]))
    averaged_data = []
    for cell_line, values in cell_line_map.items():
        q_avg = sum(v[0] for v in values) / len(values)
        f_avg = sum(v[1] for v in values) / len(values)
        averaged_data.append({"cell_line": cell_line, "quantity": q_avg, "function": f_avg})

    avg_quantity = sum(d["quantity"] for d in averaged_data) / len(averaged_data) if averaged_data else 0.0
    avg_function = sum(d["function"] for d in averaged_data) / len(averaged_data) if averaged_data else 0.0

    c_score, c_details, c_flags = assess_clinical_severity(clinical_data or {})
    c_label = describe_clinical_support(c_score, c_flags)
    explanation_parts = [
        f"functional_avg=({avg_quantity:.1f} qty, {avg_function:.1f} func)",
        f"clinical={c_label} (score={c_score})"
    ]

    # Convenience flags
    low_qty = avg_quantity < 20
    very_low_func = avg_function < 5
    high_qty = avg_quantity >= 40
    high_func = avg_function >= 60
    mid_func_iv = (10 < avg_function < 60)
    borderline_iv = (5 < avg_function <= 10)
    gating_like = (avg_function < 10 and avg_quantity >= 40)

    # Mechanistic classification (functional-priority)
    if "nonsense" in variant_type_lower and avg_quantity < 5 and avg_function < 5:
        predicted_class = "Class I (No synthesis)"
    if predicted_class is None and low_qty and very_low_func:
        predicted_class = "Class II (Defective processing)"
    if predicted_class is None and gating_like:
        predicted_class = "Class III (Defective regulation)"
    if predicted_class is None and high_qty and mid_func_iv:
        predicted_class = "Class IV (Reduced conductance)"
    if predicted_class is None and high_qty and borderline_iv:
        predicted_class = "Class IV (Reduced conductance - borderline)"
    if predicted_class is None and high_qty and high_func:
        predicted_class = "Class V (Residual function)"
    if predicted_class is None and avg_quantity < 40 and avg_function >= 40:
        predicted_class = "Class V (Residual function - low quantity)"
    if predicted_class is None:
        predicted_class = "Uncertain class (does not match known patterns)"

    # Known borderline overrides
    if variant_name in borderline_iv_v:
        predicted_class = "Class IV/V borderline"

    # Clinical reconciliation
    severe_functional = (
        predicted_class.startswith(("Class I", "Class II", "Class III"))
        or ("Class IV" in predicted_class and avg_function < 5)
    )
    if severe_functional and c_score <= 1:
        predicted_class = predicted_class + " [clinical-discordant]"
        explanation_parts.append("discordant: severe functional vs mild clinical")

    if predicted_class.startswith("Class V") and c_score >= 5:
        predicted_class = "Class IV/V borderline (clinical severe)"
        explanation_parts.append("relabel: V vs severe clinical -> IV/V borderline")

    if predicted_class.startswith("Class IV") and c_score <= 1:
        predicted_class = predicted_class + " (clinically mild)"
        explanation_parts.append("annotation: Class IV but mild clinical")

    return predicted_class, avg_quantity, avg_function, {
        "clinical_reconciliation": c_label,
        "clinical_score": c_score,
        "clinical_flags": sorted(list(c_flags)),
        "explanation": "; ".join(explanation_parts),
        "clinical_details": c_details
    }


# -----------------------------
# Main: batch across all CFTR2 variants
# -----------------------------

def robust_load_existing(path="cftr_batch_results.json"):
    if not os.path.exists(path):
        return {"completed": [], "errors": [], "run_timestamp": None}
    try:
        with open(path, encoding="utf-8") as f:
            content = f.read().strip()
            if not content:
                return {"completed": [], "errors": [], "run_timestamp": None}
            data = json.loads(content)
            # backward-friendly normalization
            if isinstance(data, list):
                return {"completed": data, "errors": [], "run_timestamp": None}
            if isinstance(data, dict):
                data.setdefault("completed", [])
                data.setdefault("errors", [])
                return data
            return {"completed": [], "errors": [], "run_timestamp": None}
    except json.JSONDecodeError:
        return {"completed": [], "errors": [], "run_timestamp": None}

def save_results_atomic(obj, path="cftr_batch_results.json"):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def main(
    excel_path="CFTR2_25September2024.xlsx",
    excel_sheet="CFTR2 variants by legacy name",
    save_folder="cftr2_scientific_pages",
    out_json="cftr_batch_results.json",
    url_log_csv="variant_url_log_scientific.csv",
    headless=False
):
    # Load all variants from Excel
    df = pd.read_excel(excel_path, sheet_name=excel_sheet, skiprows=9)
    variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]
    print(f"📄 Loaded {len(variant_names)} variants from Excel")

    # Load existing results and compute processed set
    results_obj = robust_load_existing(out_json)
    completed = results_obj.get("completed", [])
    errors = results_obj.get("errors", [])
    processed = {rec.get("variant") for rec in completed if isinstance(rec, dict)}

    # Selenium session (accept terms once)
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    # Accept terms
    try:
        print("🔐 Navigating to CFTR2 welcome page...")
        driver.get("https://cftr2.org/welcome")
        time.sleep(2)
        checkbox_ids = ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]
        for box_id in checkbox_ids:
            try:
                checkbox = wait.until(EC.presence_of_element_located((By.ID, box_id)))
                driver.execute_script("arguments[0].scrollIntoView(true);", checkbox)
                time.sleep(0.3)
                if not checkbox.is_selected():
                    driver.execute_script("arguments[0].click();", checkbox)
                print(f"✅ Checked: {box_id}")
            except Exception as e:
                print(f"⚠️ Could not check box {box_id}: {e}")

        try:
            submit_button = wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms")))
            submit_button.click()
            print("✅ Submitted agreement form.")
        except Exception as e:
            print(f"⚠️ Could not click submit button: {e}")
        time.sleep(1)
    except Exception as e:
        print(f"❌ Failed to open/accept CFTR2 terms: {e}")

    # Batch process
    os.makedirs(save_folder, exist_ok=True)
    url_log_rows = []
    failed_variants = []

    try:
        for idx, variant in enumerate(variant_names, start=1):
            v = variant.strip()
            if v in processed:
                print(f"⏭️ Skipping already processed: {v}")
                continue

            print(f"\n🧬 [{idx}/{len(variant_names)}] Processing {v}")
            try:
                # Fetch and save HTML
                html_path = fetch_scientific_page_html(driver, wait, v, save_folder)
                url_log_rows.append([v, f"https://cftr2.org/mutation/scientific/{encode_variant(v)}", "Success"])

                # Parse -> classify
                function_data, clinical_data, fev1_data, variant_type = parse_cftr2_html(html_path)
                predicted_class, avg_qty, avg_func, clinical_ctx = classify_variant(
                    function_data, clinical_data, fev1_data, variant_type, variant_name=v
                )

                record = {
                    "variant": v,
                    "variant_type": variant_type,
                    "predicted_class": predicted_class,
                    "functional_summary": {"avg_quantity": round(avg_qty, 2), "avg_function": round(avg_func, 2)},
                    "function_data": function_data,
                    "clinical_data": clinical_data,
                    "fev1_data": fev1_data,
                    "clinical_context": clinical_ctx
                }
                completed.append(record)
                processed.add(v)

                # Incremental save
                results_obj["completed"] = completed
                results_obj["errors"] = errors
                results_obj["run_timestamp"] = datetime.utcnow().isoformat() + "Z"
                save_results_atomic(results_obj, out_json)
                print(f"✅ [{v}] {predicted_class} (qty={avg_qty:.1f}, func={avg_func:.1f})")

            except Exception as e:
                print(f"❌ [{v}] Error: {e}")
                traceback.print_exc()
                failed_variants.append(v)
                errors.append({"variant": v, "error": str(e), "timestamp": datetime.utcnow().isoformat() + "Z"})
                # Save errors incrementally too
                results_obj["errors"] = errors
                results_obj["run_timestamp"] = datetime.utcnow().isoformat() + "Z"
                save_results_atomic(results_obj, out_json)
                url_log_rows.append([v, f"https://cftr2.org/mutation/scientific/{encode_variant(v)}", f"Error: {e}"])

    finally:
        driver.quit()

    # Save URL log and failed variants
    with open(url_log_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Variant", "Encoded Scientific URL", "Status"])
        writer.writerows(url_log_rows)

    if failed_variants:
        with open("failed_variants_scientific.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(failed_variants))

    print("\n🎉 Batch classification completed.")
    print(f"   ✅ Completed: {len(completed)}")
    print(f"   ⚠️ Errors: {len(errors)}")
    if failed_variants:
        print("   See failed_variants_scientific.txt for details.")
    print(f"   Results saved to: {out_json}")
    print(f"   URL log saved to: {url_log_csv}")


# -----------------------------
# Entry
# -----------------------------

if __name__ == "__main__":
    main(
        excel_path="CFTR2_25September2024.xlsx",
        excel_sheet="CFTR2 variants by legacy name",
        save_folder="cftr2_scientific_pages",
        out_json="cftr_batch_results.json",
        url_log_csv="variant_url_log_scientific.csv",
        headless=False  # set True for servers/CI
    )

In [48]:
import json
import glob
import os

# Load master file
with open("cftr_batch_results.json", encoding="utf-8") as f:
    raw_master = json.load(f)

# Normalize master data to a dict keyed by variant
if isinstance(raw_master, dict):
    master_index = raw_master
elif isinstance(raw_master, list):
    master_index = {rec["variant"]: rec for rec in raw_master if isinstance(rec, dict) and "variant" in rec}
else:
    raise ValueError("Unsupported format in master file")

# Merge updated chunks
for path in glob.glob("updated_chunks/cftr_chunk_*.json"):
    print(f"🔄 Merging updates from {path} ...")
    with open(path, encoding="utf-8") as f:
        updated_chunk = json.load(f)

    for updated_rec in updated_chunk:
        variant = updated_rec.get("variant")
        if variant:
            master_index[variant] = updated_rec

# Save merged output as a list
merged_data = list(master_index.values())
with open("cftr_batch_results_merged.json", "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=2)

print("\n✅ Merge complete. Output saved to cftr_batch_results_merged.json")



✅ Merge complete. Output saved to cftr_batch_results_merged.json


In [ ]:
import os
import re
import json
import glob
from collections import Counter, defaultdict

# ---------- Configurable thresholds ----------
MIN_MEASUREMENTS = 2   # require at least 2 (qty, func) pairs to call a mechanistic class
CLASS_V_MIN_QTY = 50   # tighten residual function
CLASS_V_MIN_FUNC = 70

# ---------- Helper checks ----------
def count_valid_measures(rec):
    frows = rec.get("function_data", []) or []
    return sum(1 for d in frows if d.get("quantity") is not None and d.get("function") is not None)

def avg_pair(rec):
    fs = rec.get("functional_summary", {}) or {}
    return fs.get("avg_quantity"), fs.get("avg_function")

def clinical_meta(rec):
    ctx = rec.get("clinical_context", {}) or {}
    return ctx.get("clinical_score"), ctx.get("clinical_reconciliation"), ctx.get("clinical_flags", [])

def is_known_splice_override(rec):
    v = (rec.get("variant") or "").strip()
    return v in {"3849+10kbC->T", "2789+5G->A", "3272-26A->G"}

def text_has(patterns, text):
    if not text:
        return False
    tl = text.lower()
    return any(p in tl for p in patterns)

# ---------- Rules producing flags ----------
def audit_record(rec):
    flags = []
    rec_class = (rec.get("predicted_class") or "").strip()
    variant = rec.get("variant")
    n_meas = count_valid_measures(rec)
    avg_q, avg_f = avg_pair(rec)
    c_score, c_label, c_flags = clinical_meta(rec)
    expl = (rec.get("clinical_context", {}) or {}).get("explanation", "")

    # 1) Insufficient data but mechanistic class assigned (except known splice overrides)
    if n_meas < MIN_MEASUREMENTS and rec_class and not rec_class.lower().startswith("uncertain"):
        if not is_known_splice_override(rec):
            flags.append({
                "code": "INSUFFICIENT_DATA_CLASS",
                "msg": f"{n_meas} valid functional measurements, but class assigned: {rec_class}"
            })

    # 2) Class V too lenient
    if rec_class.startswith("Class V"):
        if avg_q is None or avg_f is None:
            flags.append({"code":"CLASS_V_NO_AVG","msg":"Class V assigned without averages"})
        else:
            if avg_q < CLASS_V_MIN_QTY or avg_f < CLASS_V_MIN_FUNC:
                flags.append({
                    "code": "CLASS_V_WEAK",
                    "msg": f"Class V with weak functional support (qty={avg_q}, func={avg_f})"
                })

    # 3) Severe functional vs clinically mild
    if rec_class and any(rec_class.startswith(k) for k in ("Class I","Class II","Class III")):
        if c_score is not None and c_score <= 1:
            flags.append({"code":"DISCORDANT_MILD_CLINICAL","msg":f"Severe class ({rec_class}) but clinical mild (score={c_score})"})

    # 4) Class V yet clinical severe
    if rec_class.startswith("Class V") and c_score is not None and c_score >= 5:
        flags.append({"code":"V_VS_SEVERE_CLINICAL","msg":f"Class V but clinical severe (score={c_score})"})

    # 5) Class IV with very low function (<5) or clinically mild
    if rec_class.startswith("Class IV"):
        if avg_f is not None and avg_f < 5:
            flags.append({"code":"CLASS_IV_FUNC_TOO_LOW","msg":f"Class IV but function extremely low: {avg_f}"})
        if c_score is not None and c_score <= 1:
            flags.append({"code":"CLASS_IV_CLINICALLY_MILD","msg":f"Class IV but clinically mild (score={c_score})"})

    # 6) Aggregates near thresholds (edge calls)
    #    These merit human review for threshold tightening.
    if avg_q is not None and avg_f is not None:
        near = []
        def near_thr(val, thr, eps=1.0):  # within ±1
            return abs(val - thr) <= eps
        if near_thr(avg_q, 5) or near_thr(avg_f, 5):
            near.append("~5")
        if near_thr(avg_q, 10) or near_thr(avg_f, 10):
            near.append("~10")
        if near_thr(avg_q, 20) or near_thr(avg_f, 20):
            near.append("~20")
        if near_thr(avg_q, 40) or near_thr(avg_f, 40):
            near.append("~40")
        if near_thr(avg_q, CLASS_V_MIN_QTY) or near_thr(avg_f, CLASS_V_MIN_FUNC):
            near.append("~ClassV-thresholds")
        if near:
            flags.append({"code":"NEAR_THRESHOLD","msg":f"Near thresholds {','.join(near)} (qty={avg_q}, func={avg_f})"})

    # 7) Uncertain but strong/clear signal (possible upgrade)
    if rec_class.lower().startswith("uncertain") and avg_q is not None and avg_f is not None and n_meas >= MIN_MEASUREMENTS:
        if avg_q < 5 and avg_f < 5:
            flags.append({"code":"UNCERTAIN_BUT_CLASS_I","msg":"Uncertain but matches Class I signature"})
        elif avg_q < 20 and avg_f < 10:
            flags.append({"code":"UNCERTAIN_BUT_CLASS_II","msg":"Uncertain but matches Class II signature"})
        elif avg_q >= 40 and avg_f < 10:
            flags.append({"code":"UNCERTAIN_BUT_CLASS_III","msg":"Uncertain but matches Class III signature"})
        elif avg_q >= 40 and 10 <= avg_f < 60:
            flags.append({"code":"UNCERTAIN_BUT_CLASS_IV","msg":"Uncertain but matches Class IV signature"})
        elif avg_q >= CLASS_V_MIN_QTY and avg_f >= CLASS_V_MIN_FUNC:
            flags.append({"code":"UNCERTAIN_BUT_CLASS_V","msg":"Uncertain but matches tightened Class V signature"})

    return flags

def audit_chunk_file(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    flagged = []
    class_counter = Counter()
    for rec in data:
        rec_class = (rec.get("predicted_class") or "Missing").strip()
        class_counter[rec_class] += 1
        flags = audit_record(rec)
        if flags:
            flagged.append({
                "variant": rec.get("variant"),
                "predicted_class": rec_class,
                "avg_quantity": (rec.get("functional_summary") or {}).get("avg_quantity"),
                "avg_function": (rec.get("functional_summary") or {}).get("avg_function"),
                "clinical_score": (rec.get("clinical_context") or {}).get("clinical_score"),
                "flags": flags
            })
    return flagged, class_counter

def main(chunks_glob="cftr_chunk_*.json", out_dir="audit_out"):
    os.makedirs(out_dir, exist_ok=True)
    all_summary = Counter()
    index = []

    for path in sorted(glob.glob(chunks_glob)):
        print(f"🔎 Auditing {path} ...")
        flagged, counts = audit_chunk_file(path)
        all_summary.update(counts)

        base = os.path.splitext(os.path.basename(path))[0]
        out_path = os.path.join(out_dir, base + "_audit.json")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(flagged, f, indent=2)
        print(f"  → {len(flagged)} flagged entries → {out_path}")

        index.append({
            "chunk": base,
            "flagged_count": len(flagged),
            "class_counts": dict(counts)
        })

    with open(os.path.join(out_dir, "audit_index.json"), "w", encoding="utf-8") as f:
        json.dump(index, f, indent=2)
    with open(os.path.join(out_dir, "class_distribution_overall.json"), "w", encoding="utf-8") as f:
        json.dump(dict(all_summary), f, indent=2)

    print("\n✅ Audit complete.")
    print("  • audit_out/audit_index.json → per-chunk counts and flags")
    print("  • audit_out/class_distribution_overall.json → class totals across all chunks")
    print("  • audit_out/cftr_chunk_X_audit.json → only the variants that need review")

if __name__ == "__main__":
    main()
